Use Colab AI to help you develop a simple decision-tree classifier for the ppt example

Ask Colab AI "how can I create a decision tree classifier?"

# Task 1: try on PPT dataset

In [5]:
from sklearn.tree import DecisionTreeClassifier

# Create a decision tree classifier object
#clf = DecisionTreeClassifier()
clf = DecisionTreeClassifier(criterion="gini", max_depth = 5, min_samples_split = 2, min_samples_leaf = 1, random_state = 42)

# Train the classifier on the PPT dataset
X_train=[[50,80],[45,60],[19,100],[100,30],[100,70]]
y_train=["good","no good","no good","good","no good"]
clf.fit(X_train, y_train)

# Predict labels for new data
X_test=[[70,30],[60,19],[43,49]]
y_pred = clf.predict(X_test)
y_pred

array(['good', 'good', 'no good'], dtype='<U7')

how can I print the decision tree after training?

In [6]:
from sklearn.tree import export_text

# Train your decision tree classifier (as before)

# Print the decision tree
tree_text = export_text(clf)
print(tree_text)

|--- feature_0 <= 47.50
|   |--- class: no good
|--- feature_0 >  47.50
|   |--- feature_1 <= 50.00
|   |   |--- class: good
|   |--- feature_1 >  50.00
|   |   |--- feature_1 <= 75.00
|   |   |   |--- class: no good
|   |   |--- feature_1 >  75.00
|   |   |   |--- class: good



The decision tree is different to that given in the ppt, but they do the same classification task. Which one is better (simpler)?
Ocam's Razer Principle - make your model as simple (general) as possible to avoid overfitting to the training data. ==> the one obtained here may be better

# Task 2: try on Titanic dataset

In [4]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

csv_path = Path("titanic.csv")
df = pd.read_csv(csv_path)

# Handle column naming differences across Titanic CSV variants.
target_col = next((c for c in ["Survived", "survived", "2urvived"] if c in df.columns), None)
sibsp_col = next((c for c in ["SibSp", "sibsp"] if c in df.columns), None)

if target_col is None or sibsp_col is None:
    raise ValueError(f"Unexpected columns. Found: {df.columns.tolist()}")

feature_cols = ["Pclass", "Sex", "Age", sibsp_col, "Parch", "Fare", "Embarked"]

work_df = df[feature_cols + [target_col]].copy()
X = work_df[feature_cols]
y = work_df[target_col]

numeric_features = ["Pclass", "Age", sibsp_col, "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Columns in use:", feature_cols + [target_col])
print(df.head(5))
print(f"Rows: {len(df)}")

Columns in use: ['Pclass', 'Sex', 'Age', 'sibsp', 'Parch', 'Fare', 'Embarked', '2urvived']
   Passengerid   Age     Fare  Sex  sibsp  zero  zero.1  zero.2  zero.3  \
0            1  22.0   7.2500    0      1     0       0       0       0   
1            2  38.0  71.2833    1      1     0       0       0       0   
2            3  26.0   7.9250    1      0     0       0       0       0   
3            4  35.0  53.1000    1      1     0       0       0       0   
4            5  35.0   8.0500    0      0     0       0       0       0   

   zero.4  ...  zero.12  zero.13  zero.14  Pclass  zero.15  zero.16  Embarked  \
0       0  ...        0        0        0       3        0        0       2.0   
1       0  ...        0        0        0       1        0        0       0.0   
2       0  ...        0        0        0       3        0        0       2.0   
3       0  ...        0        0        0       1        0        0       2.0   
4       0  ...        0        0        0       3    

In [5]:
from sklearn.tree import DecisionTreeClassifier

clf = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "model",
            DecisionTreeClassifier(
                criterion="gini",
                max_depth=4,
                min_samples_split=10,
                min_samples_leaf=5,
                random_state=42,
            ),
        ),
    ]
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 0.7595

Confusion Matrix:
[[172  22]
 [ 41  27]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8075    0.8866    0.8452       194
           1     0.5510    0.3971    0.4615        68

    accuracy                         0.7595       262
   macro avg     0.6793    0.6418    0.6534       262
weighted avg     0.7409    0.7595    0.7456       262



In [6]:
from sklearn.tree import export_text

model = clf.named_steps["model"]
feature_names = clf.named_steps["prep"].get_feature_names_out()

tree_text = export_text(model, feature_names=list(feature_names), max_depth=4)
print(tree_text)

|--- cat__Sex_0.0 <= 0.50
|   |--- num__Pclass <= 2.50
|   |   |--- num__Age <= 42.50
|   |   |   |--- num__Age <= 29.50
|   |   |   |   |--- class: 1
|   |   |   |--- num__Age >  29.50
|   |   |   |   |--- class: 1
|   |   |--- num__Age >  42.50
|   |   |   |--- num__Fare <= 25.96
|   |   |   |   |--- class: 1
|   |   |   |--- num__Fare >  25.96
|   |   |   |   |--- class: 0
|   |--- num__Pclass >  2.50
|   |   |--- num__Fare <= 24.81
|   |   |   |--- num__Age <= 6.50
|   |   |   |   |--- class: 1
|   |   |   |--- num__Age >  6.50
|   |   |   |   |--- class: 0
|   |   |--- num__Fare >  24.81
|   |   |   |--- num__Age <= 10.50
|   |   |   |   |--- class: 0
|   |   |   |--- num__Age >  10.50
|   |   |   |   |--- class: 0
|--- cat__Sex_0.0 >  0.50
|   |--- num__Age <= 4.50
|   |   |--- num__sibsp <= 2.50
|   |   |   |--- num__Fare <= 24.50
|   |   |   |   |--- class: 1
|   |   |   |--- num__Fare >  24.50
|   |   |   |   |--- class: 1
|   |   |--- num__sibsp >  2.50
|   |   |   |--- class